Втор фајл од проектна задача -  истражување на GАТ (Graph Attention Network)
===
Graph Attention Network е невронска мрежа за графови која користи механизам на внимание за да научи кои соседни јазли се поважни за еден јазол.



> Која е разликата меѓу GAT и GCN?



> GCN ги комбинира информациите од соседните јазли со нормализирани тежини определени од структурата на графот.



> GAT учи тежини на внимание од соседните јазли, за да одреди колку информацијата од секој сосед ќе влијае врз целниот јазол.

GAT прво создава node еmbeddings со помош на информациите од соседните протеини и научените тежини на внимание. Потоа, можеме да ги користиме тие embeddings со истиот dot product декодер од претходно за да предвидиме дали два протеини се поврзани или не.

*Забелешка: Во овој фајл нема да ги објаснуваме работите од првиот фајл, само она што е ново.*







In [1]:
!pip install torch-geometric --quiet

!pip install pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-2.11.0+cu130.html --quiet

!pip install ogb --upgrade --quiet
!pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html


In [2]:
!pip show torch

Name: torch
Version: 2.11.0+cu130
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.13/dist-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu13, nvidia-cusparselt-cu13, nvidia-nccl-cu13, nvidia-nvshmem-cu13, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, fastai, ogb, peft, sentence-transformers, timm, torchdata, torchvision


In [3]:
from ogb.linkproppred import PygLinkPropPredDataset
from torch_geometric.loader import DataLoader
import torch
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage

/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [4]:
torch.serialization.add_safe_globals(
    [DataEdgeAttr, DataTensorAttr, GlobalStorage]
)

In [5]:
dataset = PygLinkPropPredDataset(name = "ogbl-ppa")

Downloaded 0.38 GB: 100%|██████████| 388/388 [00:32<00:00, 11.98it/s]


Extracting dataset/ppassoc.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00,  2.43it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 370.13it/s]


Saving...


Done!


In [6]:
print(dataset[0].x.shape)
print(dataset[0].edge_index.shape)

torch.Size([576289, 58])
torch.Size([2, 42463862])


In [7]:
import torch


if not hasattr(torch, "_real_load"):
    torch._real_load = torch.load

def trusted_load(*args, **kwargs):
    kwargs["weights_only"] = False

    return torch._real_load(*args, **kwargs)

torch.load = trusted_load




In [8]:
split_edge = dataset.get_edge_split()
graph = dataset[0]
graph.x = graph.x.float()
print(graph.x.dtype)

torch.float32


In [9]:

train_edge, valid_edge, test_edge = split_edge["train"], split_edge["valid"], split_edge["test"]

In [10]:
subset_size = 100000
train_edges_small = train_edge["edge"][:subset_size]
validation_edges_small = valid_edge["edge"][:20000]
validation_edges_neg_small = valid_edge["edge_neg"][:20000]
test_edges_small = test_edge["edge"][:20000]
test_edges_neg_small = test_edge["edge_neg"][:20000]

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
graph = graph.to(device)

Во кодот подолу се претставени 3 различни GAT модели. Единствената разлика меѓу нив е бројот на attention heads или бројот на глави за внимание.



> Што е тоа attention heads?


> Кога имаме повеќе attention heads тоа значи дека моделот користи повеќе механизми за внимание истовремено за да ги анализира соседните јазли на различни начини а, кога имаме само еден attention heads моделот користи само еден механизам за да ја процени важноста на соседите.

Во трите модели може да се забележи уште нешто, дека кога имаме „повеќе глави на внимание“ користиме concat a, кога имаме само една глава не користиме. Што е concat?


> Секој attention head ги анализира соседните јазли и создава своја репрезентација. Да речеме дека имаме 3 attention heads, а секој создава по 64 карактеристики. Кога ќе го споиме резултатот ќе имаме вкупно 192 карактеристики. Доколку ставиме concat = False ние не го зголемуваме бројот на карактеристики на 192 туку ги пресметуваме просечните вредности од сите три heads.

  Во нашиот експеримент ќе користиме concat=False од практични причини. Затоа што моделите со 1, 2 и 3 heads ќе имаат ист број на излезни карактеристики и тој број ќе биде многу помал отколку бројот на излезни карактеристики на моделите што имаат конфигурирано concat=True.








In [12]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class GAT1(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()

        self.conv1 = GATConv(
            in_channels,
            hidden_channels,
            heads=1
        )

        self.conv2 = GATConv(
            hidden_channels,
            hidden_channels,
            heads=1
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [13]:
class GAT2(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()

        self.conv1 = GATConv(
            in_channels,
            hidden_channels,
            heads=2,
            concat=False
        )

        self.conv2 = GATConv(
            hidden_channels,
            hidden_channels,
            heads=2,
            concat=False
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [14]:
class GAT3(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()

        self.conv1 = GATConv(
            in_channels,
            hidden_channels,
            heads=3,
            concat=False
        )

        self.conv2 = GATConv(
            hidden_channels,
            hidden_channels,
            heads=3,
            concat=False
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [15]:
model_configs = {
    "GAT-1-head-64": (GAT1, 64),
    "GAT-2-heads-64": (GAT2, 64),
    "GAT-3-heads-64": (GAT3, 64),
    "GAT-1-head-128": (GAT1, 128),
    "GAT-2-heads-128": (GAT2, 128),
    "GAT-3-heads-128": (GAT3, 128)
}

In [16]:

from torch_geometric.loader import LinkNeighborLoader

train_loader = LinkNeighborLoader(
    data=graph,
    num_neighbors=[5, 5],
    edge_label_index=train_edges_small.t(),
    edge_label=torch.ones(train_edges_small.size(0)),
    batch_size=256,
    shuffle=True,
    neg_sampling_ratio=1.0
)

In [17]:
from torch_geometric.loader import LinkNeighborLoader

valid_loader = LinkNeighborLoader(
    data=graph,
    num_neighbors=[5, 5],

    edge_label_index=torch.cat([
        validation_edges_small.t(),
        validation_edges_neg_small.t()
    ], dim=1),

    edge_label=torch.cat([
        torch.ones(validation_edges_small.size(0)),
        torch.zeros(validation_edges_neg_small.size(0))
    ]),
    batch_size=256,
    shuffle=False

)

In [18]:
def decode(z, edge):
    src, dst = edge.t()
    return (z[src] * z[dst]).sum(dim=1)

In [19]:
import time

def train(model, optimizer):
    model.train()

    start = time.time()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        z = model(batch.x, batch.edge_index)

        edge_label_index = batch.edge_label_index
        edge_label = batch.edge_label.float()

        score = decode(z, edge_label_index.t())

        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            score,
            edge_label
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    sec = time.time() - start

    return avg_loss, sec

In [20]:
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def evaluate(model):
    model.eval()

    all_scores = []
    all_labels = []

    for batch in valid_loader:
        batch = batch.to(device)

        z = model(batch.x, batch.edge_index)

        edge_label_index = batch.edge_label_index
        edge_label = batch.edge_label.float()

        score = decode(z, edge_label_index.t())
        prob = torch.sigmoid(score).cpu()

        all_scores.append(prob)
        all_labels.append(edge_label.cpu())

    y_score = torch.cat(all_scores)
    y_true = torch.cat(all_labels)

    auc = roc_auc_score(
        y_true.numpy(),
        y_score.numpy()
    )

    return auc

In [21]:
import torch
import time

learning_rates = [0.01, 0.001]
num_epochs = 50

results = {}

for model_name, (ModelClass, hidden_channels) in model_configs.items():

    for lr in learning_rates:

        experiment_name = f"{model_name}-lr-{lr}"

        print(f"\nTraining: {experiment_name}")

        # Create a new model
        model = ModelClass(
            in_channels=58,
            hidden_channels=hidden_channels
        ).to(device)

        # Create a new optimizer
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr
        )

        best_auc = 0
        history = []

        for epoch in range(1, num_epochs + 1):

            epoch_start = time.time()

            # Training
            loss, train_time_sec = train(model, optimizer)

            # Validation
            val_auc = evaluate(model)

            epoch_total_sec = time.time() - epoch_start

            history.append({
                "epoch": epoch,
                "train_loss": loss,
                "val_auc": val_auc,
                "train_time_sec": train_time_sec,
                "epoch_total_sec": epoch_total_sec
            })

            if val_auc > best_auc:
                best_auc = val_auc

                torch.save(
                    model.state_dict(),
                    f"best_{experiment_name}.pt"
                )

            print(
                f"Epoch {epoch:02d} | "
                f"Train Loss: {loss:.4f} | "
                f"Val AUC: {val_auc:.4f} | "
                f"Train Time: {train_time_sec:.2f}s | "
                f"Epoch Total: {epoch_total_sec:.2f}s"
            )

        results[experiment_name] = {
            "best_val_auc": best_auc,
            "history": history
        }

        print(f"Best Validation AUC: {best_auc:.4f}")


Training: GAT-1-head-64-lr-0.01
Epoch 01 | Train Loss: 0.4914 | Val AUC: 0.8981 | Train Time: 13.36s | Epoch Total: 14.63s
Epoch 02 | Train Loss: 0.4796 | Val AUC: 0.9006 | Train Time: 6.53s | Epoch Total: 7.75s
Epoch 03 | Train Loss: 0.4798 | Val AUC: 0.9041 | Train Time: 6.60s | Epoch Total: 7.84s
Epoch 04 | Train Loss: 0.4786 | Val AUC: 0.9077 | Train Time: 6.55s | Epoch Total: 7.78s
Epoch 05 | Train Loss: 0.4768 | Val AUC: 0.9136 | Train Time: 6.62s | Epoch Total: 7.85s
Epoch 06 | Train Loss: 0.4763 | Val AUC: 0.9164 | Train Time: 6.92s | Epoch Total: 8.65s
Epoch 07 | Train Loss: 0.4732 | Val AUC: 0.9188 | Train Time: 6.48s | Epoch Total: 7.68s
Epoch 08 | Train Loss: 0.4729 | Val AUC: 0.9174 | Train Time: 6.55s | Epoch Total: 7.77s
Epoch 09 | Train Loss: 0.4723 | Val AUC: 0.9154 | Train Time: 6.46s | Epoch Total: 7.66s
Epoch 10 | Train Loss: 0.4719 | Val AUC: 0.9134 | Train Time: 6.41s | Epoch Total: 7.65s
Epoch 11 | Train Loss: 0.4716 | Val AUC: 0.9178 | Train Time: 6.55s | Epoch

 Од претходниот код во кој ги трениравме моделите и им вршевме валидација имавме променлива results која беше речник. И за секој експеримент (вкупно 12 - 6 модели кои се тренираат со 2 вида на оптимизери = 12) ние чуваме речник внатре во главниот речник results.

На пример доколку имавме само два експерименти нашиот results речник ќе изгледаше вака:



    results = {
      "GAT-1-head-64-lr-0.01": {
          "best_val_auc": 0.85,
          "history": [
              {"epoch": 1, "train_loss": 0.65, "val_auc": 0.72},
              {"epoch": 2, "train_loss": 0.58, "val_auc": 0.78},
              {"epoch": 3, "train_loss": 0.52, "val_auc": 0.85}
          ]
      },

      "GAT-2-heads-64-lr-0.01": {
          "best_val_auc": 0.88,
          "history": [
              {"epoch": 1, "train_loss": 0.62, "val_auc": 0.75},
              {"epoch": 2, "train_loss": 0.54, "val_auc": 0.82},
              {"epoch": 3, "train_loss": 0.49, "val_auc": 0.88}
          ]
      }
    }


Секое име на експеримент е различен клуч во речникот results. Секој клуч има вредност но, во нашиот случај вредноста е нов речник (под-речник).

> best_val_auc е клуч во подречникот кој ја содржи најдобрата вредност под ROC кривата на валидациското множество за секој експеримент.



> history е листа која содржи информации за секоја епоха.






In [38]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

PyTorch: 2.11.0+cu130
CUDA: 13.0


In [39]:
for name, result in results.items():
    print(f"{name}: {result['best_val_auc']:.4f}")

GAT-1-head-64-lr-0.01: 0.9289
GAT-1-head-64-lr-0.001: 0.9293
GAT-2-heads-64-lr-0.01: 0.9322
GAT-2-heads-64-lr-0.001: 0.9323
GAT-3-heads-64-lr-0.01: 0.9354
GAT-3-heads-64-lr-0.001: 0.9351
GAT-1-head-128-lr-0.01: 0.9256
GAT-1-head-128-lr-0.001: 0.9331
GAT-2-heads-128-lr-0.01: 0.9309
GAT-2-heads-128-lr-0.001: 0.9371
GAT-3-heads-128-lr-0.01: 0.9357
GAT-3-heads-128-lr-0.001: 0.9383


In [40]:
import pandas as pd
import zipfile
from google.colab import files

history_rows = []

for experiment_name, result in results.items():
    for epoch_data in result["history"]:
        history_rows.append({
            "experiment": experiment_name,
            **epoch_data
        })

gat_history = pd.DataFrame(history_rows)
gat_history.to_csv("gat_history.csv", index=False)

with zipfile.ZipFile("gat_best_models.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    for experiment_name in results:
        model_file = f"best_{experiment_name}.pt"
        zipf.write(model_file)

files.download("gat_history.csv")
files.download("gat_best_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Последниот код ќе го објасниме подетално затоа што во претходниот фајл со GCN моделите не е објаснет.

Најпрвин иницијализираме празна листа која се вика history_rows. Потоа се извршуваат двата for циклуса еден во друг.

    for experiment_name, result in results.items():
    for epoch_data in result["history"]:
        history_rows.append({
            "experiment": experiment_name,
            **epoch_data
        })



>  Првиот for циклус го минува целиот речник „results“. Односно „experiment_name“ значи името на експериментот (кој модел + кој оптимизер го користиме) а, „result“ е речникот што ги содржи информациите best_val_auc и history за тој експеримент.



> Сега за вториот циклус го земаме речникот „result“ и од него ги земаме вредностите за клучот „history“. Тој клуч содржи вредности за секоја епоха.

Односно, двата циклуси заедно минуваат низ сите 12 експерименти и за секој експеримент ги изминуваат вредностите од секоја епоха. Сега кога интерираме низ сите епохи во сите експерименти во листата „history_rows“ додаваме речници. Тие речници ги содржат следните клучеви:



*   Име на експериментот
*   Сите клучеви што се наоѓаат во речникот history

Нормално во „history_rows“ се додаваат и соодветните вредности на овие клучеви.







> gat_history = pd.DataFrame(history_rows)

По завршувањето на овие два циклуса, листата history_rows ја претвораме во табела.



> gat_history.to_csv("gat_history.csv", index=False)

Оваа табела ја претвораме во .csv фајл и со „index=False“ кажуваме дека не сакаме дополнителна колона со индексите на редовите.




    with zipfile.ZipFile("gat_best_models.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
        for experiment_name in results:
            model_file = f"best_{experiment_name}.pt"
            zipf.write(model_file)

Овој код креира еден zip фајл кој ги содржи 12-те најдобри GAT модели. Ајде да ја објасниме првата линија:



> Командата zipfile.ZipFile() креира или отвора zip архива. Првиот аргумент во неа е името на zip фајлот а, вториот е write mode - креира нов zip фајл или брише и повторно запишува во веќе постоечки. Третиот аргумент пак, ZIP_DEFLATED значи дека фајлот ќе биде компресиран за да му ја намалиме големината.

Сега продолжуваме во циклусот:


> Ги листаме сите експерименти во речникот results (вкупно се 12) и креираме променлива model_file која на имињата на експериментите додава „best_“ како префикс и „.pt“ како суфикс. Овие имиња ние ги имаме и претходно користено кога го трениравме моделот со:

     torch.save(    model.state_dict(),
                    f"best_{experiment_name}.pt")

  ние ги зачувавме параметрите на моделите локално во Colab но, за да ги имам локално на мојот компјутер, ние правиме: zipf.write(model_file) кои го наоѓа постоечкиот фајл кој се чува во Colab за да го додаде во zip архивата.







    files.download("gat_history.csv")
    files.download("gat_best_models.zip")

На крајот само ги симнуваме фајловите локално на мојот компјутер.

In [41]:
from torch_geometric.loader import LinkNeighborLoader

test_loader = LinkNeighborLoader(
    data=graph,
    num_neighbors=[5, 5],

    edge_label_index=torch.cat([
        test_edges_small.t(),
        test_edges_neg_small.t()
    ], dim=1),

    edge_label=torch.cat([
        torch.ones(test_edges_small.size(0)),
        torch.zeros(test_edges_neg_small.size(0))
    ]),

    batch_size=256,
    shuffle=False
)

In [42]:
gat_models = {}

for model_name, (ModelClass, hidden_channels) in model_configs.items():

    for lr in learning_rates:

        experiment_name = f"{model_name}-lr-{lr}"


        model = ModelClass(
            in_channels=58,
            hidden_channels=hidden_channels
        ).to(device)


        model.load_state_dict(
            torch.load(
                f"best_{experiment_name}.pt",
                map_location=device,
                weights_only=True
            )
        )


        model.eval()


        gat_models[experiment_name] = model

        print(f"Loaded: {experiment_name}")

Loaded: GAT-1-head-64-lr-0.01
Loaded: GAT-1-head-64-lr-0.001
Loaded: GAT-2-heads-64-lr-0.01
Loaded: GAT-2-heads-64-lr-0.001
Loaded: GAT-3-heads-64-lr-0.01
Loaded: GAT-3-heads-64-lr-0.001
Loaded: GAT-1-head-128-lr-0.01
Loaded: GAT-1-head-128-lr-0.001
Loaded: GAT-2-heads-128-lr-0.01
Loaded: GAT-2-heads-128-lr-0.001
Loaded: GAT-3-heads-128-lr-0.01
Loaded: GAT-3-heads-128-lr-0.001


In [43]:
from sklearn.metrics import roc_auc_score
import torch

@torch.no_grad()
def test(model):
    model.eval()

    all_scores = []
    all_labels = []

    for batch in test_loader:
        batch = batch.to(device)

        z = model(batch.x, batch.edge_index)

        edge_label_index = batch.edge_label_index
        edge_label = batch.edge_label.float()

        score = decode(z, edge_label_index.t())

        prob = torch.sigmoid(score)

        all_scores.append(prob.cpu())
        all_labels.append(edge_label.cpu())

    all_scores = torch.cat(all_scores)
    all_labels = torch.cat(all_labels)

    auc = roc_auc_score(
        all_labels.numpy(),
        all_scores.numpy()
    )

    return auc

In [44]:
test_results = {}

for experiment_name, model in gat_models.items():

    test_auc = test(model)

    test_results[experiment_name] = test_auc

    print(f"{experiment_name}: Test AUC = {test_auc:.4f}")

GAT-1-head-64-lr-0.01: Test AUC = 0.9279
GAT-1-head-64-lr-0.001: Test AUC = 0.9291
GAT-2-heads-64-lr-0.01: Test AUC = 0.9323
GAT-2-heads-64-lr-0.001: Test AUC = 0.9325
GAT-3-heads-64-lr-0.01: Test AUC = 0.9345
GAT-3-heads-64-lr-0.001: Test AUC = 0.9319
GAT-1-head-128-lr-0.01: Test AUC = 0.9233
GAT-1-head-128-lr-0.001: Test AUC = 0.9312
GAT-2-heads-128-lr-0.01: Test AUC = 0.9306
GAT-2-heads-128-lr-0.001: Test AUC = 0.9355
GAT-3-heads-128-lr-0.01: Test AUC = 0.9334
GAT-3-heads-128-lr-0.001: Test AUC = 0.9363


In [45]:
import pandas as pd
from google.colab import files

final_rows = []

for experiment_name, result in results.items():

    final_rows.append({
        "model": experiment_name,
        "best_val_auc": result["best_val_auc"],
        "test_auc": test_results[experiment_name],
        "epochs": 50
    })

final_metrics = pd.DataFrame(final_rows)

final_metrics.to_csv("gat_final_metrics.csv", index=False)

files.download("gat_final_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>